In [1]:
import os

In [2]:
pwd%

'D:\\ML and AI Engineer\\Projects\\An-end-to-end-customer-churn-prediction-model\\research'

In [3]:
# os.chdir("..")

In [5]:
import shap
import joblib
import numpy as np
import pandas as pd
from src.customer_churn.exception.exception import CustomerChurnException
from src.customer_churn.logging.logger import logging

In [45]:
%pwd

'D:\\ML and AI Engineer\\Projects\\An-end-to-end-customer-churn-prediction-model'

# Creating the `ShapExplainer` class:

In [40]:
class ShapExplainer:
    
    def __init__(self, model, preprocessor):
        self.model = model
        self.preprocessor = preprocessor
        self.explainer = shap.TreeExplainer(model)

    
    def get_feature_names(self) -> list:
        """
        Get feature names after preprocessing.
        """
        try:
            num_transformer = self.preprocessor.named_transformers_['numerical']
            cat_transformer = self.preprocessor.named_transformers_['categorical']
            num_names = (
                num_transformer.get_feature_names_out().tolist()
                if hasattr(num_transformer, 'get_feature_names_out')
                else self.preprocessor.transformers[0][2]
            )
            cat_names = cat_transformer.get_feature_names_out().tolist()
            return num_names + cat_names
        except Exception as e:
            raise CustomerChurnException(e, sys)


    def _get_feature_value(self, feature, X_raw):
        """
        Get original feature value from raw input.
        """
        try:
            val = X_raw.iloc[0][feature]
            if isinstance(val, float):
                return round(val, 4)
            return val
        except Exception:
            return None
            

    def _get_shap_row(self, shap_values):
        """
        Handle different SHAP output formats.
        """
        if isinstance(shap_values, list):
            return shap_values[1][0]
        elif len(shap_values.shape) == 2:
            return shap_values[0]
        else:
            return shap_values
            

    def _aggregate_one_hot(self, contributions):
        """
        Aggregate ONLY one-hot encoded categorical features.
        """
        grouped = {}
        cat_transformer = self.preprocessor.named_transformers_['categorical']
        cat_feature_names = cat_transformer.feature_names_in_
    
        for feature, value in contributions:    
            matched = False
            for cat_col in cat_feature_names:
                prefix = f"{cat_col}_"
    
                if feature.startswith(prefix):
                    grouped[cat_col] = grouped.get(cat_col, 0.0) + value
                    matched = True
                    break
            if not matched:
                grouped[feature] = grouped.get(feature, 0.0) + value
    
        return sorted(
            grouped.items(),
            key=lambda x: abs(x[1]),
            reverse=True
        )
    
    
    def explain(self, X_raw: pd.DataFrame) -> dict:
        """
        Takes raw features and returns SHAP explanations.
        """
        try:
            X_transformed = self.preprocessor.transform(X_raw)
            shap_values = self.explainer.shap_values(X_transformed)
            feature_names = self.get_feature_names()
            shap_row = self._get_shap_row(shap_values)
            raw_contributions = list(zip(feature_names, shap_row))
            raw_sorted = sorted(
                raw_contributions,
                key=lambda x: abs(x[1]),
                reverse=True
            )
            aggregated = self._aggregate_one_hot(raw_contributions)
            return {
                "shap_values": shap_row.tolist(),
                "feature_names": feature_names,
                "raw_contributions": [
                    {"feature": f, "shap_value": round(float(v), 4)}
                    for f, v in raw_sorted
                ],
               "aggregated_contributions": [
                                                {
                                                    "feature": f,
                                                    "value": self._get_feature_value(f, X_raw),
                                                    "shap_value": round(float(v), 4)
                                                }
                                                for f, v in aggregated
                                            ]
            }
        except Exception as e:
            raise CustomerChurnException(e, sys)

# Tesing on the first feature of the `X.parquet` data 

In [41]:
test = pd.read_parquet("Artifacts/pre_processing/full_dataset/X.parquet")
feature = test.head(1)

In [42]:
feature

,order_frequency,total_monetary_value,total_quantity_abs,total_order_issues,country,avg_order_value,avg_quantity_per_order,customer_order_issue_rate,recency,tenure,avg_stockcode_issue_rate,max_stockcode_issue_rate,total_orders_made_for_stock
0,11,-180.33,62,12,United Kingdom,-16.393636,5.636364,1.090909,177,471,0.124901,1.026667,5464.0


In [43]:
model = joblib.load("Artifacts/model_training/final_model.pkl")
preprocessor = joblib.load("Artifacts/pre_processing/pre_processor.pkl")
shap_explainer = ShapExplainer(model=model, preprocessor=preprocessor)
results = shap_explainer.explain(X_raw=feature)

In [44]:
results['aggregated_contributions']

[{'feature': 'order_frequency', 'value': 11, 'shap_value': -0.5208},
 {'feature': 'total_monetary_value', 'value': -180.33, 'shap_value': 0.1497},
 {'feature': 'total_orders_made_for_stock',
  'value': 5464.0,
  'shap_value': -0.0899},
 {'feature': 'avg_order_value', 'value': -16.3936, 'shap_value': -0.0847},
 {'feature': 'customer_order_issue_rate',
  'value': 1.0909,
  'shap_value': 0.0775},
 {'feature': 'avg_quantity_per_order', 'value': 5.6364, 'shap_value': -0.0629},
 {'feature': 'recency', 'value': 177, 'shap_value': 0.039},
 {'feature': 'total_order_issues', 'value': 12, 'shap_value': -0.0295},
 {'feature': 'avg_stockcode_issue_rate',
  'value': 0.1249,
  'shap_value': 0.0259},
 {'feature': 'tenure', 'value': 471, 'shap_value': -0.0213},
 {'feature': 'max_stockcode_issue_rate',
  'value': 1.0267,
  'shap_value': -0.0211},
 {'feature': 'total_quantity_abs', 'value': 62, 'shap_value': -0.0114},
 {'feature': 'country', 'value': 'United Kingdom', 'shap_value': 0.0043}]